# Delta Table

In [0]:
%sql
create table man_cata.man_Schema.deltatbl
(
  id int,
  name string,
  city string
)
using delta
location 'abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/deltalake/deltatbl'

In [0]:
%sql
alter table man_cata.man_schema.deltatbl set tblproperties ('delta.enableDeletionVectors'= false)

In [0]:
%sql
insert into man_cata.man_schema.deltatbl values (1, 'aa','delhi'), (2, 'bb', 'mumbai'), (3, 'cc', 'chennai')

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
describe extended man_cata.man_schema.deltatbl

col_name,data_type,comment
id,int,null
name,string,null
city,string,null
,,
# Delta Statistics Columns,,
Column Names,"id, name, city",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,man_cata,


In [0]:
%sql
select * from man_cata.man_schema.deltatbl

id,name,city
1,aa,delhi
2,bb,mumbai
3,cc,chennai


**Updates in delta table**

In [0]:
%sql
update man_cata.man_schema.deltatbl
set city = 'bangalore' where id = 1

num_affected_rows
1


In [0]:
%sql
select * from man_cata.man_schema.deltatbl

id,name,city
1,aa,bangalore
2,bb,mumbai
3,cc,chennai


##### Once you perform any crud operation on the delta table it will generate a new parquet file and tombstoned the previous one. in the _delta_log folder it will also create another json file with all the info

##### in the latest json file it will version the partition according to our crud operation, this called versioning 
##### command to check the versioning

In [0]:
%sql
describe history man_cata.man_schema.deltatbl

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2025-03-12T10:20:25Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,UPDATE,"Map(predicate -> [""(id#2003 = 1)""])",null,List(3443715516544330),0311-142743-rpp3vq22,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1115, numCopiedRows -> 2, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1846, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1047, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1125, rewriteTimeMs -> 773)",null,Databricks-Runtime/14.3.x-scala2.12
2,2025-03-12T10:09:07Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3443715516544330),0311-142743-rpp3vq22,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1115)",null,Databricks-Runtime/14.3.x-scala2.12
1,2025-03-12T10:07:30Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableDeletionVectors"":""false""})",null,List(3443715516544330),0311-142743-rpp3vq22,0,WriteSerializable,true,Map(),null,Databricks-Runtime/14.3.x-scala2.12
0,2025-03-12T10:03:21Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,CREATE TABLE,"Map(partitionBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3443715516544330),0311-142743-rpp3vq22,null,WriteSerializable,true,Map(),null,Databricks-Runtime/14.3.x-scala2.12


#### Time Travel
##### you can restore the previous data using time travel.

In [0]:
%sql
RESTORE man_cata.man_schema.deltatbl TO VERSION AS OF 2

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1115,1,1,1,1125,1115


In [0]:
%sql
select * from man_cata.man_schema.deltatbl

id,name,city
1,aa,delhi
2,bb,mumbai
3,cc,chennai


### Deletion Vector

In [0]:
%sql
create table man_cata.man_Schema.deltatbl2
(
  id int,
  name string,
  city string
)
using delta
location 'abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/deltalake/deltatbl2'

In [0]:
%sql
insert into man_cata.man_schema.deltatbl2 values (1, 'aa','delhi'), (2, 'bb', 'mumbai'), (3, 'cc', 'chennai')

num_affected_rows,num_inserted_rows
3,3


**updates in deletion vector table**

In [0]:
%sql
alter table man_cata.man_schema.deltatbl set tblproperties ('delta.enableDeletionVectors'= true)

In [0]:
%sql
update man_cata.man_schema.deltatbl2
set city = 'bbsr' where id = 1

num_affected_rows
1


##### once you performed crud it will create a deletion_vector .bin file and 2 new partitions. deletion_vector will contain the deleted data and inside the delta log json it will remove the data then add the same add with the updated one to tell us which one should it remove. one more partition has been created using optimize command by delta lake to combine all the files to make one file


## Optimize in delta table

#### it alwasy easier to read 4gb of one file instead of 1gb of 4 files

In [0]:
%sql
describe history man_cata.man_schema.deltatbl2

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2025-03-12T10:37:32Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], zOrderBy -> [], batchId -> 0, auto -> true)",null,List(3443715516544330),0311-142743-rpp3vq22,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2199, p25FileSize -> 1108, numDeletionVectorsRemoved -> 1, minFileSize -> 1108, numAddedFiles -> 1, maxFileSize -> 1108, p75FileSize -> 1108, p50FileSize -> 1108, numAddedBytes -> 1108)",null,Databricks-Runtime/14.3.x-scala2.12
2,2025-03-12T10:37:26Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,UPDATE,"Map(predicate -> [""(id#5681 = 1)""])",null,List(3443715516544330),0311-142743-rpp3vq22,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3513, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1728, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1084, rewriteTimeMs -> 1733)",null,Databricks-Runtime/14.3.x-scala2.12
1,2025-03-12T10:35:18Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3443715516544330),0311-142743-rpp3vq22,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1115)",null,Databricks-Runtime/14.3.x-scala2.12
0,2025-03-12T10:35:15Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,CREATE TABLE,"Map(partitionBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3443715516544330),0311-142743-rpp3vq22,null,WriteSerializable,true,Map(),null,Databricks-Runtime/14.3.x-scala2.12


In [0]:
%sql
optimize man_cata.man_schema.deltatbl2

path,metrics
abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/deltalake/deltatbl2,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, 0, 1, 1, true, 0, 0, 1741776890233, 1741776892409, 4, 0, null, List(0, 0), 3, 3, 0, 0, null)"


# Deep clone Vs Shallow Clone

#### In deep clone it clones metadata + data
#### In Shallow clone it clones metadata

#### Deep Clone

In [0]:
%sql
create table man_cata.man_schema.deepclonetbl
deep clone man_cata.man_schema.deltatbl

source_table_size,source_num_of_files,num_removed_files,num_copied_files,removed_files_size,copied_files_size
1115,1,0,1,0,1115


In [0]:
%sql
select * from man_cata.man_schema.deepclonetbl

id,name,city
1,aa,delhi
2,bb,mumbai
3,cc,chennai


In [0]:
%sql
select * from man_cata.man_schema.deltatbl

id,name,city
1,aa,delhi
2,bb,mumbai
3,cc,chennai


In [0]:
%sql
update man_cata.man_schema.deltatbl set city = 'bbsr' where id = 1

num_affected_rows
1


In [0]:
%sql
select * from man_cata.man_schema.deepclonetbl

id,name,city
1,aa,delhi
2,bb,mumbai
3,cc,chennai


In [0]:
%sql
select * from man_cata.man_schema.deltatbl

id,name,city
1,aa,bbsr
2,bb,mumbai
3,cc,chennai


In [0]:
%sql
describe history man_cata.man_schema.deepclonetbl

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2025-03-12T11:14:33Z,6636894047528416,ashutoshacharya21_gmail.com#ext#@ashutoshacharya21gmail.onmicrosoft.com,CLONE,"Map(source -> man_cata.man_schema.deltatbl, sourceVersion -> 4, isShallow -> false)",null,List(3443715516544330),0311-142743-rpp3vq22,-1,Serializable,false,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 1115, numCopiedFiles -> 1, copiedFilesSize -> 1115, sourceNumOfFiles -> 1)",null,Databricks-Runtime/14.3.x-scala2.12


In [0]:
%sql
describe extended man_cata.man_schema.deepclonetbl

col_name,data_type,comment
id,int,null
name,string,null
city,string,null
,,
# Delta Statistics Columns,,
Column Names,"id, name, city",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,man_cata,


#### Shallow Clone

In [0]:
%sql
create table man_cata.man_schema.shallowtbl
shallow clone man_cata.man_schema.man_table

source_table_size,source_num_of_files,num_removed_files,num_copied_files,removed_files_size,copied_files_size
0,0,0,0,0,0


In [0]:
%sql
describe extended man_cata.man_schema.shallowtbl

col_name,data_type,comment
id,int,null
name,string,null
,,
# Detailed Table Information,,
Catalog,man_cata,
Database,man_schema,
Table,shallowtbl,
Created Time,Wed Mar 12 11:22:41 UTC 2025,
Last Access,UNKNOWN,
Created By,Spark,
